Instead of teaching the model to classify an image into a category, Im going to teach it to project an image into a 256-dimensional vector space. 

Goal: force similar paintings to have vectors that are close together, and dissimilar paintings to have vectors that are far apart.

Model choice:

Deep Metric Learning with Siamese Triplet Network

Why?
- Optimal for Content-Based Image Retrieval (CBIR), compared to standard approaches

- Siamese Network drops the classification head. Instead, acts as a spatial projector

- Triplet Loss
    - standard contrastive loss (comparing two images at a time), the network might just learn to output the exact same vector for every portrait, collapsing the space

    - Triplet Margin Loss prevents this by comparing three images simultaneously

    - forces the network to ensure the Positive is closer to the Anchor than the Negative ($n$) by a strictly defined mathematical margin ($\alpha$)

    $$\mathcal{L} = \max(d(a, p) - d(a, n) + \alpha, 0)$$


In [23]:
import pandas as pd
import numpy as np 
import os

import torch
import torch.nn as nn
import torchvision
from PIL import Image
import random

### Dataset

In [24]:
data_df = pd.read_csv("./nga_dataset/nga_paintings_metadata.csv")
print(data_df.columns)

Index(['objectid', 'uuid_x', 'accessioned', 'accessionnum', 'locationid',
       'title', 'displaydate', 'beginyear', 'endyear', 'visualbrowsertimespan',
       'medium', 'dimensions', 'inscription', 'markings',
       'attributioninverted', 'attribution', 'provenancetext', 'creditline',
       'classification', 'subclassification', 'visualbrowserclassification',
       'parentid', 'isvirtual', 'departmentabbr', 'portfolio', 'series',
       'volume', 'watermarks', 'lastdetectedmodification', 'wikidataid',
       'customprinturl', 'uuid_y', 'iiifurl', 'iiifthumburl', 'viewtype',
       'sequence', 'width', 'height', 'maxpixels', 'openaccess', 'created',
       'modified', 'depictstmsobjectid', 'assistivetext'],
      dtype='str')


In [25]:
portrait_keywords = [
    'portrait', 'head', 'face', 'woman', 'man', 'boy', 'girl', 'madonna', 
    'saint', 'lady', 'gentleman', 'child', 'king', 'queen', 'duke', 
    'self-portrait', 'family', 'figure', 'nude', 'bather', 'peasant', 
    'soldier', 'christ', 'virgin', 'angel', 'apostle', 'pope', 'doge'
]

landscape_keywords = [
    'landscape', 'river', 'mountain', 'valley', 'view', 'scene', 'sea', 
    'beach', 'coast', 'forest', 'wood', 'tree', 'field', 'meadow', 
    'waterfall', 'lake', 'pond', 'harbor', 'bay', 'ruins', 'cityscape', 
    'bridge', 'canal', 'farm', 'sunset', 'sunrise'
]

still_life_keywords = [
    'still life', 'flowers', 'fruit', 'bowl', 'vase', 'basket', 'glass', 
    'table', 'plate', 'dish', 'bottle', 'cup', 'bread', 'cheese', 'wine', 
    'meat', 'fish', 'books', 'skull', 'bouquet', 'peaches', 'apples', 'grapes'
]

def pseudo_labeling(title):
    if not isinstance(title,str): return -1

    title = title.lower()

    if any(word in title for word in portrait_keywords): return 0
    elif any(word in title for word in landscape_keywords): return 1
    elif any(word in title for word in still_life_keywords): return 2
    else: return -1

data_df['pseudo_label'] = data_df['title'].apply(pseudo_labeling)

labeled_df = data_df[data_df['pseudo_label'] != -1].copy()
print(f"Paintings successfully pseudo-labeled: {len(labeled_df)}")

img_dir = "nga_dataset/images"
labeled_df['img_path'] = labeled_df['objectid'].apply(lambda x: os.path.join(img_dir,f"{x}.jpg"))

final_df = labeled_df[labeled_df['img_path'].apply(os.path.exists)].copy()
print(f"Final usable dataset size: {len(final_df)}")

print("\nClass Distribution (0: Portrait, 1: Landscape, 2: Still Life):")
print(final_df['pseudo_label'].value_counts())

Paintings successfully pseudo-labeled: 1670
Final usable dataset size: 1670

Class Distribution (0: Portrait, 1: Landscape, 2: Still Life):
pseudo_label
0    1065
1     400
2     205
Name: count, dtype: int64


In [26]:
class TripletDataset(torch.utils.data.Dataset):
    def __init__(self,df,transform=None):
        self.df = df
        self.transform = transform

        self.portraits = df[df['pseudo_label']==0]['img_path'].tolist()
        self.landscapes = df[df['pseudo_label']==1]['img_path'].tolist()
        self.still_life = df[df['pseudo_label']==2]['img_path'].tolist()

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        anchor_path = row['img_path']
        anchor_label = row['pseudo_label']
        
        positive_list,negative_list=[],[]

        if(anchor_label==0):
            positive_list = self.portraits
            negative_list = self.landscapes + self.still_life
        elif(anchor_label==1):
            positive_list = self.landscapes
            negative_list = self.portraits + self.still_life
        else:
            positive_list = self.still_life
            negative_list = self.portraits + self.landscapes
        
        pos_path = random.choice(positive_list)
        neg_path = random.choice(negative_list)
        while(pos_path==anchor_path): pos_path = random.choice(positive_list)

        anchor_img = Image.open(anchor_path).convert('RGB')
        pos_img = Image.open(pos_path).convert('RGB')
        neg_img = Image.open(neg_path).convert('RGB')

        anchor_img = self.transform(anchor_img)
        pos_img = self.transform(pos_img)
        neg_img = self.transform(neg_img)

        return anchor_img,pos_img,neg_img
        


In [27]:
import torchvision.transforms as T

train_transform = T.Compose(
    [
        T.Resize(256),
        T.RandomCrop(224),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

dataset = TripletDataset(df=final_df,transform=train_transform)
anchor,pos,neg = dataset[0]
print(f"Anchor shape: {anchor.shape}")
print(f"Positive shape: {pos.shape}")
print(f"Negative shape: {neg.shape}")

Anchor shape: torch.Size([3, 224, 224])
Positive shape: torch.Size([3, 224, 224])
Negative shape: torch.Size([3, 224, 224])
